# 04 — Drift Analysis

Computes **all drift metrics** from Chapter 3 of the thesis for each window pair (A, B) and produces the full temporal reporting.

**Input:** `data/processed/`, `data/windows/`, `data/models/`, `data/shap/`  
**Output:** `data/results/drift_metrics.csv` and figures in `data/results/`

---

| Metric | Symbol | Section |
|--------|--------|---------|
| Covariate drift | Δ_X | §3.2 |
| Target drift | Δ_Y | §3.2 |
| Performance change | Δ_perf | §3.2 |
| Local dynamic drift (cosine) | Δ_E^{loc}(cos) | §3.5 |
| Local dynamic drift (RBO) | Δ_E^{loc}(rbo) | §3.5 |
| Within-window baseline (cosine) | σ_E^{pool}(cos) | §3.6 |
| Within-window baseline (RBO) | σ_E^{pool}(rbo) | §3.6 |
| Drift Ratio (cosine) | DR_cos | §3.7 |
| Drift Ratio (RBO) | DR_rbo | §3.7 |
| Global SHAP drift | Δ_E^{glob} | §3.8 |

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from itertools import combinations

# RBO implementation
try:
    import rbo
    HAS_RBO = True
except ImportError:
    HAS_RBO = False
    print('WARNING: rbo package not found. RBO metrics will be skipped.')
    print('Install with: pip install rbo')

WORKSPACE  = Path(r'C:\Users\Frewl\OneDrive - KU Leuven\KU Leuven\Thesis\Shoppers_workspace')
PROC_DIR   = WORKSPACE / 'data' / 'processed'
WIN_DIR    = WORKSPACE / 'data' / 'windows'
MODEL_DIR  = WORKSPACE / 'data' / 'models'
SHAP_DIR   = WORKSPACE / 'data' / 'shap'
RESULTS_DIR = WORKSPACE / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RBO_P = 0.9   # persistence parameter for RBO
EPS   = 1e-8  # small constant for Drift Ratio denominator

print('Imports OK')

In [ ]:
X    = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']
n_features = len(feature_names)

print(f'X: {X.shape}, features: {n_features}, R={R}, pairs: {len(pairs)}')

## Distance and RBO helper functions

In [ ]:
def cosine_distance(u: np.ndarray, v: np.ndarray) -> float:
    """d_cos(u,v) = 1 - cosine_similarity(u,v). Returns 0 for zero vectors."""
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u < 1e-12 or norm_v < 1e-12:
        return 0.0 if (norm_u < 1e-12 and norm_v < 1e-12) else 1.0
    return float(1.0 - np.dot(u, v) / (norm_u * norm_v))


def rbo_distance(u: np.ndarray, v: np.ndarray, p: float = RBO_P) -> float:
    """
    d_rbo(u, v) = 1 - RBO(rank(u), rank(v)).
    Features are ranked by decreasing absolute attribution.
    """
    if not HAS_RBO:
        return np.nan
    # Rank by decreasing |value| → list of feature indices
    rank_u = list(np.argsort(-np.abs(u)))
    rank_v = list(np.argsort(-np.abs(v)))
    score = rbo.RankingSimilarity(rank_u, rank_v).rbo(p=p)
    return float(1.0 - score)


def instance_dynamic_drift(
    phi_A: np.ndarray,  # (R, p) — R attribution vectors from window A
    phi_B: np.ndarray,  # (R, p) — R attribution vectors from window B
    dist_fn,
) -> float:
    """
    δ_dyn(x; d) = mean over all R×R cross-window pairs of d(φ_A^{(r)}, φ_B^{(s)}).
    """
    dists = []
    for r in range(R):
        for s in range(R):
            dists.append(dist_fn(phi_A[r], phi_B[s]))
    return float(np.mean(dists))


def instance_baseline_instability(
    phi: np.ndarray,  # (R, p) — R attribution vectors from same window
    dist_fn,
) -> float:
    """
    δ^{base}(x; d) = median over all r < r' pairs of d(φ^{(r)}, φ^{(r')}).
    """
    dists = []
    for r, r2 in combinations(range(R), 2):
        dists.append(dist_fn(phi[r], phi[r2]))
    return float(np.median(dists)) if dists else 0.0


def rbo_global_drift(
    phi_bar_A: np.ndarray,  # (|F|, p)
    phi_bar_B: np.ndarray,  # (|F|, p)
    p: float = RBO_P,
) -> float:
    """
    Δ_E^{glob}: 1 − RBO(rank(g_A), rank(g_B)) where g(j) = mean_x |φ̄_j(x)|.
    """
    if not HAS_RBO:
        return np.nan
    g_A = np.abs(phi_bar_A).mean(axis=0)
    g_B = np.abs(phi_bar_B).mean(axis=0)
    rank_A = list(np.argsort(-g_A))
    rank_B = list(np.argsort(-g_B))
    score = rbo.RankingSimilarity(rank_A, rank_B).rbo(p=p)
    return float(1.0 - score)


print('Distance functions defined.')

## Main drift computation loop

For each window pair we compute all metrics and append to a results list.

In [ ]:
results = []

for p in pairs:
    pid      = p['pair_id']
    pair_dir = MODEL_DIR / f'pair_{pid:02d}'
    shap_dir = SHAP_DIR  / f'pair_{pid:02d}'

    print(f'\n── Pair {pid:02d}: A_end={p["step_label_A"]}  B_end={p["step_label_B"]} ──')

    # ── Load data ────────────────────────────────────────────────────
    scaler    = joblib.load(pair_dir / 'scaler.joblib')
    pred_data = np.load(pair_dir / 'predictions.npz')
    shap_A    = np.load(shap_dir / 'shap_A.npy')   # (R, |F|, p)
    shap_B    = np.load(shap_dir / 'shap_B.npy')

    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_B    = np.array(p['idx_B'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)
    flagged_local_idx = pred_data['flagged_idx']

    n_flagged = len(flagged_local_idx)
    print(f'  Flagged: {n_flagged:,}')

    row = {
        'pair_id':       pid,
        'step_label_A':  p['step_label_A'],
        'step_label_B':  p['step_label_B'],
        'n_train_A':     p['n_train_A'],
        'n_train_B':     p['n_train_B'],
        'n_eval':        p['n_eval'],
        'n_flagged':     n_flagged,
        'pr_auc_A':      float(pred_data['pr_auc_A']),
        'pr_auc_B':      float(pred_data['pr_auc_B']),
    }

    # ── §3.2 Performance change ───────────────────────────────────────
    loss_A = 1.0 - float(pred_data['pr_auc_A'])
    loss_B = 1.0 - float(pred_data['pr_auc_B'])
    row['delta_perf'] = loss_A - loss_B   # positive → B is better

    # ── §3.2 Target drift ─────────────────────────────────────────────
    mean_Y_A = Y[idx_A].mean()
    mean_Y_B = Y[idx_B].mean()
    row['delta_Y'] = abs(mean_Y_A - mean_Y_B)
    print(f'  Target drift: {row["delta_Y"]:.4f}  (base rate A={mean_Y_A:.4f}, B={mean_Y_B:.4f})')

    # ── §3.2 Covariate drift (Wasserstein, feature-averaged) ──────────
    X_A_raw = X[idx_A]   # unscaled for fair comparison
    X_B_raw = X[idx_B]
    # Scale both with the reference scaler (fitted on W_k)
    X_A_sc = scaler.transform(X_A_raw)
    X_B_sc = scaler.transform(X_B_raw)
    w1_per_feat = np.array([
        wasserstein_distance(X_A_sc[:, j], X_B_sc[:, j])
        for j in range(n_features)
    ])
    row['delta_X'] = float(w1_per_feat.mean())
    print(f'  Covariate drift: {row["delta_X"]:.4f}')

    if n_flagged == 0:
        for key in ['loc_cos', 'loc_rbo', 'base_cos_A', 'base_cos_B', 'base_rbo_A',
                    'base_rbo_B', 'sigma_cos', 'sigma_rbo',
                    'drift_ratio_cos', 'drift_ratio_rbo', 'global_rbo']:
            row[key] = np.nan
        results.append(row)
        print('  No flagged instances — drift metrics set to NaN.')
        continue

    # ── §3.5 Dynamic cross-window drift ───────────────────────────────
    # φ_A: (R, |F|, p),  φ_B: (R, |F|, p)
    # For each flagged instance x_i:
    #   δ_dyn(x_i; cos) = mean_{r,s} d_cos(φ_A^{(r)}[i], φ_B^{(s)}[i])
    #   δ_dyn(x_i; rbo) = mean_{r,s} d_rbo(φ_A^{(r)}[i], φ_B^{(s)}[i])

    dyn_cos_per_instance = []
    dyn_rbo_per_instance = []

    for i in range(n_flagged):
        phi_A_i = shap_A[:, i, :]   # (R, p)
        phi_B_i = shap_B[:, i, :]   # (R, p)
        dyn_cos_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, cosine_distance))
        dyn_rbo_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, rbo_distance))

    dyn_cos_arr = np.array(dyn_cos_per_instance)
    dyn_rbo_arr = np.array(dyn_rbo_per_instance)

    row['loc_cos'] = float(np.median(dyn_cos_arr))
    row['loc_rbo'] = float(np.nanmedian(dyn_rbo_arr))
    print(f'  Loc drift (cos): {row["loc_cos"]:.4f}  (rbo): {row["loc_rbo"]:.4f}')

    # ── §3.6 Within-window baseline instability ───────────────────────
    base_cos_A_per = []
    base_cos_B_per = []
    base_rbo_A_per = []
    base_rbo_B_per = []

    for i in range(n_flagged):
        phi_A_i = shap_A[:, i, :]   # (R, p)
        phi_B_i = shap_B[:, i, :]   # (R, p)
        base_cos_A_per.append(instance_baseline_instability(phi_A_i, cosine_distance))
        base_cos_B_per.append(instance_baseline_instability(phi_B_i, cosine_distance))
        base_rbo_A_per.append(instance_baseline_instability(phi_A_i, rbo_distance))
        base_rbo_B_per.append(instance_baseline_instability(phi_B_i, rbo_distance))

    sigma_A_cos = float(np.median(base_cos_A_per))
    sigma_B_cos = float(np.median(base_cos_B_per))
    sigma_A_rbo = float(np.nanmedian(base_rbo_A_per))
    sigma_B_rbo = float(np.nanmedian(base_rbo_B_per))

    row['base_cos_A'] = sigma_A_cos
    row['base_cos_B'] = sigma_B_cos
    row['base_rbo_A'] = sigma_A_rbo
    row['base_rbo_B'] = sigma_B_rbo
    row['sigma_cos']  = 0.5 * (sigma_A_cos + sigma_B_cos)   # pooled
    row['sigma_rbo']  = 0.5 * (sigma_A_rbo + sigma_B_rbo)
    print(f'  Baseline (cos): σ_A={sigma_A_cos:.4f}, σ_B={sigma_B_cos:.4f}, pooled={row["sigma_cos"]:.4f}')
    print(f'  Baseline (rbo): σ_A={sigma_A_rbo:.4f}, σ_B={sigma_B_rbo:.4f}, pooled={row["sigma_rbo"]:.4f}')

    # ── §3.7 Drift Ratio ──────────────────────────────────────────────
    row['drift_ratio_cos'] = row['loc_cos'] / (row['sigma_cos'] + EPS)
    row['drift_ratio_rbo'] = row['loc_rbo'] / (row['sigma_rbo'] + EPS)
    print(f'  Drift Ratio (cos): {row["drift_ratio_cos"]:.3f}  (rbo): {row["drift_ratio_rbo"]:.3f}')

    # ── §3.8 Global SHAP drift ────────────────────────────────────────
    phi_bar_A = shap_A.mean(axis=0)   # (|F|, p)
    phi_bar_B = shap_B.mean(axis=0)
    row['global_rbo'] = rbo_global_drift(phi_bar_A, phi_bar_B)
    print(f'  Global SHAP drift (rbo): {row["global_rbo"]:.4f}')

    results.append(row)

print('\n✓ All drift metrics computed.')

## Save results

In [ ]:
drift_df = pd.DataFrame(results)
drift_df.to_csv(RESULTS_DIR / 'drift_metrics.csv', index=False)
print(f'Saved drift_metrics.csv ({len(drift_df)} rows)')
print(drift_df[[
    'pair_id', 'step_label_A', 'step_label_B',
    'pr_auc_A', 'pr_auc_B', 'delta_perf',
    'delta_X', 'delta_Y',
    'loc_cos', 'sigma_cos', 'drift_ratio_cos',
    'loc_rbo', 'sigma_rbo', 'drift_ratio_rbo',
    'global_rbo'
]].to_string(index=False))

## §3.10 Reporting: time-series plots

In [ ]:
df = drift_df.dropna(subset=['loc_cos'])   # skip pairs with no flagged instances
x  = df['pair_id'].values
x_labels = [f"{r['step_label_A']}" for _, r in df.iterrows()]

fig = plt.figure(figsize=(14, 14))
gs  = gridspec.GridSpec(4, 1, hspace=0.45)

# ── Panel 1: contextual signals ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1b = ax1.twinx()
ax1.plot(x, df['delta_X'],    'o-', color='steelblue', label='Cov. drift (Δ_X)')
ax1.plot(x, df['delta_Y'],    's--', color='darkorange', label='Target drift (Δ_Y)')
ax1b.plot(x, df['delta_perf'], '^:', color='green', label='Perf. change (Δ_perf)')
ax1.set_ylabel('Drift magnitude')
ax1b.set_ylabel('Perf. change', color='green')
ax1.set_title('Contextual signals over time')
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc='upper left', fontsize=8)
ax1.set_xticks(x); ax1.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

# ── Panel 2: local dynamic drift (cosine) + baseline ──────────────────
ax2 = fig.add_subplot(gs[1])
ax2.plot(x, df['loc_cos'],   'o-',  color='crimson',  label='Loc. drift Δ_E^{loc} (cos)')
ax2.plot(x, df['sigma_cos'], 's--', color='gray',     label='Baseline σ_E^{pool} (cos)')
ax2.set_ylabel('Cosine distance')
ax2.set_title('Local explanation drift vs. within-window baseline (cosine)')
ax2.legend(fontsize=8)
ax2.set_xticks(x); ax2.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

# ── Panel 3: drift ratios ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.plot(x, df['drift_ratio_cos'], 'o-',  color='crimson',    label='Drift Ratio (cos)')
if HAS_RBO:
    ax3.plot(x, df['drift_ratio_rbo'], 's--', color='darkorchid', label='Drift Ratio (rbo)')
ax3.axhline(y=1.0, color='black', linestyle=':', linewidth=1.5, label='DR = 1 (baseline level)')
ax3.set_ylabel('Drift Ratio')
ax3.set_title('Drift Ratio (values > 1 indicate supra-baseline temporal drift)')
ax3.legend(fontsize=8)
ax3.set_xticks(x); ax3.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

# ── Panel 4: global SHAP drift ────────────────────────────────────────
ax4 = fig.add_subplot(gs[3])
if HAS_RBO:
    ax4.plot(x, df['global_rbo'], 'D-', color='teal', label='Global SHAP drift (rbo)')
ax4.set_ylabel('1 − RBO')
ax4.set_title('Global SHAP feature importance drift')
ax4.legend(fontsize=8)
ax4.set_xticks(x); ax4.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)
ax4.set_xlabel('Window A end date')

plt.suptitle('Temporal Stability of XAI — Shoppers Dataset', fontsize=13, y=1.01)
plt.savefig(RESULTS_DIR / 'temporal_drift_report.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## Per-feature SHAP drift profile

Shows which features shift most in global importance across the timeline.

In [ ]:
# Collect global importance g_A for each pair
global_imp_matrix = []  # (n_pairs, n_features)
pair_labels = []

for p in pairs:
    pid      = p['pair_id']
    shap_dir = SHAP_DIR / f'pair_{pid:02d}'
    shap_A_path = shap_dir / 'shap_A.npy'
    if not shap_A_path.exists():
        continue
    shap_A = np.load(shap_A_path)        # (R, |F|, p)
    phi_bar_A = shap_A.mean(axis=0)      # (|F|, p)
    g_A = np.abs(phi_bar_A).mean(axis=0) # (p,)
    global_imp_matrix.append(g_A)
    pair_labels.append(p['step_label_A'])

if global_imp_matrix:
    imp_arr = np.array(global_imp_matrix)   # (n_pairs, p)

    # Top 15 most variable features across time
    feat_variance = imp_arr.std(axis=0)
    top_feat_idx = np.argsort(-feat_variance)[:15]

    fig, ax = plt.subplots(figsize=(12, 5))
    for j in top_feat_idx:
        ax.plot(range(len(pair_labels)), imp_arr[:, j], marker='o', label=feature_names[j], linewidth=1.5)
    ax.set_title('Global SHAP importance over time — top 15 most variable features')
    ax.set_xlabel('Window pair (A end date)')
    ax.set_ylabel('Mean |SHAP|')
    ax.set_xticks(range(len(pair_labels)))
    ax.set_xticklabels(pair_labels, rotation=30, ha='right', fontsize=7)
    ax.legend(loc='upper right', fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'feature_importance_over_time.png', dpi=150)
    plt.show()

## Summary statistics

In [ ]:
numeric_cols = [
    'pr_auc_A', 'pr_auc_B', 'delta_perf',
    'delta_X', 'delta_Y',
    'loc_cos', 'sigma_cos', 'drift_ratio_cos',
    'loc_rbo', 'sigma_rbo', 'drift_ratio_rbo',
    'global_rbo'
]
summary = drift_df[numeric_cols].describe().T[['mean', 'std', 'min', 'max']]
print('Overall summary:')
print(summary.round(4).to_string())

# High-level interpretation
dr_cos_mean = drift_df['drift_ratio_cos'].mean()
print(f'\n--- Interpretation ---')
print(f'Mean Drift Ratio (cosine): {dr_cos_mean:.3f}')
if dr_cos_mean > 2.0:
    print('  → Temporal explanation drift is substantially larger than retraining noise.')
elif dr_cos_mean > 1.0:
    print('  → Temporal drift moderately exceeds retraining baseline.')
else:
    print('  → Temporal drift is within the range of ordinary retraining noise.')